In [1]:
import numpy as np
from itertools import product
from generative_model import GenerativeModel
from t_maze_world import TMazeWorld
from active_inference_agent import ActiveInferenceAgent
np.set_printoptions(
    linewidth=200,       # Längere Zeilen
    precision=3,         # Nur 3 Nachkommastellen anzeigen
    suppress=True        # Verhindert die wissenschaftliche Notation (z.B. 1e-05 wird zu 0.000)
)

T = 3
NUM_TRIALS = 1

ALPHA = 64
BETA = 4
PARAM_A = 0.9
PARAM_C = 2.0


def get_policies_for_step(t, num_actions=4, T=T):
    remaining_steps = T - t
    if remaining_steps <= 0:
        return [[]]
    policy_list = list(product(range(num_actions), repeat=remaining_steps))
    
    return [list(p) for p in policy_list], policy_list

print("Erstelle Simulations-Objekte...")
model = GenerativeModel(a=PARAM_A, c=PARAM_C)
world = TMazeWorld(model=model)
agent = ActiveInferenceAgent(model=model, alpha=ALPHA, beta=BETA, T=T)
print("Erstellung der Simulations-Objekte abgeschlossen")

Erstelle Simulations-Objekte...
Erstellung der Simulations-Objekte abgeschlossen


In [2]:
print(f"Starte Simulation für {NUM_TRIALS} Trials...")
print("-" * 30)
trial_results = []
for trial in range(NUM_TRIALS):
    print(f"--- Starte Trial {trial + 1}/{NUM_TRIALS} ---")
    o_t = world.reset()
    agent.reset()

    for t in range(T):
        policies_t, policy_list = get_policies_for_step(t)
        a_t, pi_hat_stable = agent.infer_action(o_t, policies_t, t)
        
        o_t_index = np.argmax(o_t)
        print(f"  t={t}: Agent sieht {o_t_index} und wählt Aktion {a_t}")
        if a_t is None:
            print("  t={t}: T erreicht. Keine Aktion mehr.")
            break
            
        o_t = world.step(a_t)

        u_prob = np.zeros(4)
        for i in range(len(policy_list)):
            u = policy_list[i][0]
            temp_u_prob = u_prob[u] + pi_hat_stable[i]
            u_prob[u] = temp_u_prob
        print(u_prob)


print("-" * 30)
print("Simulation beendet.")

Starte Simulation für 1 Trials...
------------------------------
--- Starte Trial 1/1 ---
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
gamma 6.009094194072509
  t=0: Agent sieht 1 und wählt Aktion 1
[0.217 0.215 0.215 0.352]
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
gamma 8.376798464684848
  t=1: Agent sieht 6 und wählt Aktion 1
[0.25 0.25 0.25 0.25]
gamma 16.0
gamma 16.0
gamma 16.0
gamma 16.0
gamma 16.0
gamma 16.0
gamma 16.0
gamma 16.0
  t=2: Agent sieht 1 und wählt Aktion 3
[0.25 0.25 0.25 0.25]
------------------------------
Simulation beendet.
